# 14 — LayoutLMv3 Field Extraction Playground

Load the trained LayoutLMv3 model and experiment with cleaning rules
on raw model predictions — no retraining, no generative AI.

| | |
|---|---|
| **Model** | Best checkpoint from Notebook 12 |
| **Task** | Inspect raw predictions, iterate on cleaning rules |
| **No retraining** | Weights loaded from disk, read-only |

**Workflow:**
1. Run Cells 1–4 once to load everything (takes ~5 sec)
2. Cell 5 shows raw model output on test set examples
3. Cell 6 is your sandbox — edit `clean_output()` and re-run freely
4. Cell 7 lets you test on your own images from disk
5. Cell 8 shows token-level debug when something looks wrong


## 1. Environment — run this first after every kernel restart

In [1]:
import os, time

# Must be set before any transformers/tokenizers import
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

t0 = time.time()
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor
print('transformers imported in', round(time.time() - t0, 2), 'sec')
print('Python:', __import__('sys').executable)


/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers imported in 3.27 sec
Python: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/bin/python


## 2. Paths, labels and dataset

In [2]:
import json
import torch
from pathlib import Path
from PIL import Image
from datasets import load_from_disk

PROJECT_ROOT = Path('..').resolve()
DATASET_DIR  = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
CKPT_PATH    = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'
MAX_LENGTH   = 512

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

with open(DATASET_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

BIO_LABELS = [id2label[i] for i in range(len(id2label))]

FIELD_ORDER = [
    'INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE',
    'ISSUER_NAME',    'RECIPIENT_NAME', 'TOTAL_AMOUNT'
]

raw_dataset = load_from_disk(str(DATASET_DIR))

print(f'Device     : {DEVICE}')
print(f'Checkpoint : {CKPT_PATH}')
print(f'Exists     : {CKPT_PATH.exists()}')
print(f'Labels     : {BIO_LABELS}')
print(f'Dataset    : {raw_dataset}')


Device     : mps
Checkpoint : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/layoutlmv3_fatura/best_checkpoint
Exists     : True
Labels     : ['O', 'B-INVOICE_NUMBER', 'I-INVOICE_NUMBER', 'B-INVOICE_DATE', 'I-INVOICE_DATE', 'B-DUE_DATE', 'I-DUE_DATE', 'B-ISSUER_NAME', 'I-ISSUER_NAME', 'B-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'B-TOTAL_AMOUNT', 'I-TOTAL_AMOUNT']
Dataset    : DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})


## 3. Load model and processor from checkpoint

In [3]:
# Loads from disk only — no network, no retraining needed.

processor = LayoutLMv3Processor.from_pretrained(
    str(CKPT_PATH),
    apply_ocr=False,
    use_fast=True,
    local_files_only=True,
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(CKPT_PATH),
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
)
model.to(DEVICE)
model.eval()

print('Model loaded OK')
print('  tokenizer :', type(processor.tokenizer).__name__)
print('  num labels:', model.config.num_labels)
print('  device    :', DEVICE)


The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100%|██████████| 216/216 [00:00<00:00, 8951.56it/s]


Model loaded OK
  tokenizer : LayoutLMv3Tokenizer
  num labels: 13
  device    : mps


## 4. Core inference functions — do not edit this cell

In [4]:
def get_raw_predictions(image, words, bboxes):
    """
    Run LayoutLMv3 on image + OCR words/boxes.
    Returns dict {FIELD_NAME: raw_string} with NO cleaning applied.
    This is the pure model output — the ground truth of what the model learned.
    """
    encoding = processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = model(**{k: v.to(DEVICE) for k, v in encoding.items()})

    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids    = encoding.word_ids(batch_index=0)

    # Map subword tokens back to word level (take first subword per word)
    word_preds = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]

    aligned_words    = [words[i]      for i in sorted(word_preds)]
    aligned_pred_ids = [word_preds[i] for i in sorted(word_preds)]

    # Group BIO tokens into field strings
    fields, current_field, current_tokens = {}, None, []
    for label_id, word in zip(aligned_pred_ids, aligned_words):
        label = id2label[label_id]
        if label == 'O':
            if current_field:
                text = ' '.join(current_tokens).strip()
                if text:
                    fields[current_field] = text
                current_field, current_tokens = None, []
        elif label.startswith('B-'):
            if current_field:
                text = ' '.join(current_tokens).strip()
                if text:
                    fields[current_field] = text
            current_field, current_tokens = label[2:], [word]
        elif label.startswith('I-'):
            fn = label[2:]
            if current_field == fn:
                current_tokens.append(word)
            elif current_field is None and fn in fields:
                fields[fn] += ' ' + word
            elif current_field is None:
                current_field, current_tokens = fn, [word]
            else:
                text = ' '.join(current_tokens).strip()
                if text:
                    fields[current_field] = text
                current_field, current_tokens = fn, [word]
    if current_field:
        text = ' '.join(current_tokens).strip()
        if text:
            fields[current_field] = text

    return fields


def token_debug(image, words, bboxes):
    """Show every non-O token prediction — useful for debugging."""
    encoding = processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = model(**{k: v.to(DEVICE) for k, v in encoding.items()})
    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids    = encoding.word_ids(batch_index=0)
    word_preds  = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]

    print(f"{'LABEL':<28}  WORD")
    print('-' * 55)
    for wi in sorted(word_preds):
        label = id2label[word_preds[wi]]
        if label != 'O':
            print(f"  {label:<28}: {repr(words[wi])}")


print('Core inference functions defined OK')


Core inference functions defined OK


## 5. Raw model output on test set examples

No cleaning applied — this is exactly what the model produces.
Change `N_EXAMPLES` or `SPLIT` as needed.


In [5]:
N_EXAMPLES = 5      # how many examples to show
SPLIT      = 'test' # 'train', 'val', or 'test'

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    bboxes  = example['bboxes']

    raw = get_raw_predictions(image, words, bboxes)

    print(f"\n{'='*60}")
    print(f"  {SPLIT.upper()} {i}: {Path(example['image_path']).stem}")
    print('='*60)
    for field in FIELD_ORDER:
        value = raw.get(field, '—')
        print(f"  {field:<20}: {value}")



  TEST 0: Template1_Instance189
  INVOICE_NUMBER      : —
  INVOICE_DATE        : Date 29-Apr-2012
  DUE_DATE            : Due Date 07-Aug-2010
  ISSUER_NAME         : —
  RECIPIENT_NAME      : Bill to Shelly Rodriguez 02547 Ramos Bypass Suite 849 Williamshaven, NC 38767 US Tel +(463)893-0347 Email christina14@example.org Site http //www.gomez.com/
  TOTAL_AMOUNT        : TOTAL 134.41 USD

  TEST 1: Template38_Instance29
  INVOICE_NUMBER      : INVOICE # 9Y1M9d-232
  INVOICE_DATE        : Date 15-Feb-1993
  DUE_DATE            : Due Date 14-Jan-2007
  ISSUER_NAME         : —
  RECIPIENT_NAME      : —
  TOTAL_AMOUNT        : BALANCE DUE 220.90 EUR

  TEST 2: Template28_Instance30
  INVOICE_NUMBER      : —
  INVOICE_DATE        : Invoice Date 13-Aug-2002
  DUE_DATE            : —
  ISSUER_NAME         : —
  RECIPIENT_NAME      : Bill to Daniel Moore 04274 Claudia Fort Suite 045 Patriciashire, SD 32054 US Tel +(491)040-2728 Email frichardson@example.org Site https //berger-bailey.com/
  

## 6. Cleaning sandbox — edit freely and re-run

This is your workbench. Edit `clean_output()` until the output looks right.
Re-run this cell as many times as you want — no kernel restart needed.

The table shows **RAW** (what the model said) and **CLEANED** (after your function)
side by side so you can see exactly what each rule is doing.


In [6]:
import re

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EDIT THIS FUNCTION — everything else in this cell is display code
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATE_RE = re.compile(
    r'\d{1,2}[-/][A-Za-z]{3,9}[-/]\d{2,4}'
    r'|\d{1,2}[-/]\d{1,2}[-/]\d{2,4}'
    r'|\d{4}[-/]\d{2}[-/]\d{2}'
)

AMOUNT_RE = re.compile(
    r'\d[\d,\.]+\s*(?:USD|EUR|GBP|CAD|AUD)'
    r'|(?:USD|EUR|GBP|CAD|AUD)\s*\d[\d,\.]+'  
    r'|[$€£]\s*\d[\d,\.]+',
    re.IGNORECASE
)

NAME_STOP       = {'tel','phone','fax','email','e-mail','site','web',
                   'http','https','www','address','addr','gstin','gst','vat'}
NAME_LABEL_WORDS = {'bill','to','billed','sold','send','from','by'}


def clean_output(field, raw_value):
    """
    Clean the raw model output for a given field.
    raw_value: the exact string the model produced (no pre-processing)
    Returns:   the cleaned string to display
    """
    if not raw_value or raw_value == '—':
        return '—'

    v = raw_value.strip()

    # ── Dates: find the date pattern, ignore surrounding words ────────────
    if field in ('INVOICE_DATE', 'DUE_DATE'):
        m = DATE_RE.search(v)
        return m.group(0).strip() if m else v

    # ── Amounts: find number + currency ───────────────────────────────────
    if field == 'TOTAL_AMOUNT':
        m = AMOUNT_RE.search(v)
        if m:
            return m.group(0).strip()
        m = re.search(r'\d[\d,\.]+', v)
        return m.group(0).strip() if m else v

    # ── Invoice number: take the last token ───────────────────────────────
    if field == 'INVOICE_NUMBER':
        tokens = v.split()
        return tokens[-1] if tokens else v

    # ── Names: skip label words, stop at address ──────────────────────────
    if field in ('ISSUER_NAME', 'RECIPIENT_NAME'):
        tokens = v.split()
        # Skip leading label words: 'Bill', 'to', 'From', etc.
        start = 0
        while start < len(tokens) and tokens[start].lower().strip(':.,-') in NAME_LABEL_WORDS:
            start += 1
        name = []
        for tok in tokens[start:]:
            t = tok.lower().strip(':.,-')
            if any(t.startswith(s) for s in NAME_STOP):  # address keyword
                break
            if re.match(r'^\d+$', t):                    # house number
                break
            if len(name) >= 4:                            # max name length
                break
            name.append(tok)
        return ' '.join(name).strip(',:. ') if name else v

    return v


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Display — shows RAW vs CLEANED side by side
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

N_EXAMPLES = 5
SPLIT      = 'test'

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    bboxes  = example['bboxes']

    raw = get_raw_predictions(image, words, bboxes)

    print(f"\n{'='*75}")
    print(f"  {SPLIT.upper()} {i}: {Path(example['image_path']).stem}")
    print('='*75)
    print(f"  {'FIELD':<20}  {'RAW MODEL OUTPUT':<32}  CLEANED")
    print(f"  {'-'*20}  {'-'*32}  {'-'*25}")
    for field in FIELD_ORDER:
        raw_val     = raw.get(field, '—')
        cleaned_val = clean_output(field, raw_val)
        raw_display = (raw_val[:30] + '..') if len(raw_val) > 32 else raw_val
        print(f"  {field:<20}  {raw_display:<32}  {cleaned_val}")



  TEST 0: Template1_Instance189
  FIELD                 RAW MODEL OUTPUT                  CLEANED
  --------------------  --------------------------------  -------------------------
  INVOICE_NUMBER        —                                 —
  INVOICE_DATE          Date 29-Apr-2012                  29-Apr-2012
  DUE_DATE              Due Date 07-Aug-2010              07-Aug-2010
  ISSUER_NAME           —                                 —
  RECIPIENT_NAME        Bill to Shelly Rodriguez 02547..  Shelly Rodriguez
  TOTAL_AMOUNT          TOTAL 134.41 USD                  134.41 USD

  TEST 1: Template38_Instance29
  FIELD                 RAW MODEL OUTPUT                  CLEANED
  --------------------  --------------------------------  -------------------------
  INVOICE_NUMBER        INVOICE # 9Y1M9d-232              9Y1M9d-232
  INVOICE_DATE          Date 15-Feb-1993                  15-Feb-1993
  DUE_DATE              Due Date 14-Jan-2007              14-Jan-2007
  ISSUER_NAME        

## 7. Test on your own images from disk

Put your image paths in `MY_IMAGES` and run.
Requires Cell 6 to have been run first (so `clean_output` is defined).


In [7]:
import pytesseract

# ── Put your image paths here ──────────────────────────────────────────────
MY_IMAGES = [
    '/path/to/your/invoice1.jpg',   # <- replace with real paths
    '/path/to/your/invoice2.png',
]


def ocr_image(image):
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    words, boxes = [], []
    w, h = image.size
    for i, text in enumerate(data['text']):
        text = str(text).strip()
        if not text or int(data['conf'][i]) < 10:
            continue
        left, top     = data['left'][i], data['top'][i]
        width, height = data['width'][i], data['height'][i]
        x0 = int(max(0, min(1000, round(left / w * 1000))))
        y0 = int(max(0, min(1000, round(top  / h * 1000))))
        x1 = int(max(0, min(1000, round((left + width)  / w * 1000))))
        y1 = int(max(0, min(1000, round((top  + height) / h * 1000))))
        if x1 <= x0: x1 = min(1000, x0 + 1)
        if y1 <= y0: y1 = min(1000, y0 + 1)
        words.append(text)
        boxes.append([x0, y0, x1, y1])
    return (words or ['empty']), (boxes or [[0, 0, 1, 1]])


for img_path in MY_IMAGES:
    p = Path(img_path)
    print(f"\n{'='*60}")
    print(f"  {p.name}")
    print('='*60)

    if not p.exists():
        print(f"  File not found: {img_path}")
        continue

    image        = Image.open(p).convert('RGB')
    words, boxes = ocr_image(image)
    print(f"  OCR found {len(words)} words")

    raw = get_raw_predictions(image, words, boxes)

    print(f"  {'FIELD':<20}  {'CLEANED':<25}  RAW")
    print(f"  {'-'*20}  {'-'*25}  {'-'*30}")
    for field in FIELD_ORDER:
        raw_val     = raw.get(field, '—')
        cleaned_val = clean_output(field, raw_val)
        raw_display = (raw_val[:28] + '..') if len(raw_val) > 30 else raw_val
        print(f"  {field:<20}  {cleaned_val:<25}  {raw_display}")



  invoice1.jpg
  File not found: /path/to/your/invoice1.jpg

  invoice2.png
  File not found: /path/to/your/invoice2.png


## 8. Token-level debug

Shows exactly which words the model tagged with which labels.
Use this when a field is missing or contains unexpected text.


In [8]:
# ── Config ────────────────────────────────────────────────────────────────
SPLIT             = 'test'
EXAMPLE_IDX       = 0
USE_CUSTOM_IMAGE  = False
CUSTOM_IMAGE_PATH = '/path/to/your/invoice.jpg'

# ── Load ───────────────────────────────────────────────────────────────────
if USE_CUSTOM_IMAGE:
    image        = Image.open(CUSTOM_IMAGE_PATH).convert('RGB')
    words, boxes = ocr_image(image)
    name         = Path(CUSTOM_IMAGE_PATH).name
else:
    example = raw_dataset[SPLIT][EXAMPLE_IDX]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    boxes   = example['bboxes']
    name    = Path(example['image_path']).stem

print(f'Token predictions for: {name}')
print(f'Total OCR words      : {len(words)}')
print()
token_debug(image, words, boxes)


Token predictions for: Template1_Instance189
Total OCR words      : 98

LABEL                         WORD
-------------------------------------------------------
  B-RECIPIENT_NAME            : 'Bill'
  I-RECIPIENT_NAME            : 'to'
  I-RECIPIENT_NAME            : 'Shelly'
  I-RECIPIENT_NAME            : 'Rodriguez'
  I-RECIPIENT_NAME            : '02547'
  I-RECIPIENT_NAME            : 'Ramos'
  I-RECIPIENT_NAME            : 'Bypass'
  I-RECIPIENT_NAME            : 'Suite'
  I-RECIPIENT_NAME            : '849'
  I-RECIPIENT_NAME            : 'Williamshaven,'
  I-RECIPIENT_NAME            : 'NC'
  I-RECIPIENT_NAME            : '38767'
  I-RECIPIENT_NAME            : 'US'
  I-RECIPIENT_NAME            : 'Tel'
  I-RECIPIENT_NAME            : '+(463)893-0347'
  I-RECIPIENT_NAME            : 'Email'
  I-RECIPIENT_NAME            : 'christina14@example.org'
  I-RECIPIENT_NAME            : 'Site'
  I-RECIPIENT_NAME            : 'http'
  I-RECIPIENT_NAME            : '//www.gomez.com/'
